# Data Engineering

**OULAD Student Clustering · ENSIA · Spring 2025–2026**

This notebook is the first step of our pipeline. We load the seven raw OULAD tables, merge them on each student registration, and export a single **`master_raw.csv`**. We include **all modules and presentations**  and normalize time-based engagement by module length so cross-module comparisons remain meaningful. Downstream notebooks (EDA, features, clustering) all start from this file.



## 1. Setup

We import the project helpers from `src/data_loader.py` and set paths to `data/raw` and `data/processed`. Shared functions keep this notebook aligned with what the rest of the pipeline (and the web backend) will call later.



In [29]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Repo root (parent of notebooks/)
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loader import (
    add_module_length,
    build_assessment_aggregations,
    build_vle_aggregations,
    load_oulad,
    save_master_raw,
    STUDENT_KEYS,
    MAX_WEEK,
)

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print(f"Project root: {ROOT}")


Project root: D:\Ensia\3rd year\Afaf\S2\Machine Learning\project


## 2. Load raw OULAD files

We load every CSV from `data/raw/` and record its shape. Published OULAD releases differ slightly in row counts, so we rely on the files on disk rather than fixed numbers from older documentation.



In [30]:
data = load_oulad(RAW)

courses = data["courses"]
assessments = data["assessments"]
vle = data["vle"]
student_info = data["student_info"]
student_reg = data["student_reg"]
student_assess = data["student_assess"]
student_vle = data["student_vle"]

tables = {
    "courses": courses,
    "assessments": assessments,
    "vle": vle,
    "student_info": student_info,
    "student_reg": student_reg,
    "student_assess": student_assess,
    "student_vle": student_vle,
}

for name, df in tables.items():
    print(f"\n{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
    display(df.head(3))

n_anchor = len(student_info)
print(f"\nAnchor table (student_info): {n_anchor:,} registrations")



courses: 22 rows × 3 cols


,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268



assessments: 206 rows × 6 cols


,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19.00,10.00
1,AAA,2013J,1753,TMA,54.00,20.00
2,AAA,2013J,1754,TMA,117.00,20.00



vle: 6,364 rows × 6 cols


...


student_assess: 173,912 rows × 5 cols


,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78.00
1,1752,28400,22,0,70.00
2,1752,31604,17,0,72.00



student_vle: 10,655,280 rows × 6 cols


,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1



Anchor table (student_info): 32,593 registrations


## 3. Anchor table and join keys

`studentInfo` defines our grain: **one row per student per module registration**. All other tables link through `(id_student, code_module, code_presentation)`.

**Important:** In `studentAssessment`, no row for a given assessment means the student did not submit , that is behavioral information, not a data-quality gap.



In [31]:
master = student_info.copy()
master = add_module_length(master, courses)

assert master["module_length"].notna().all(), "Some students lack module_length"
print(f"module_length range: {master['module_length'].min():.0f} – {master['module_length'].max():.0f} days")
print(master[["code_module", "code_presentation", "module_length"]].drop_duplicates().head(8))


module_length range: 234 – 269 days
      code_module code_presentation  module_length
0             AAA             2013J            268
383           AAA             2014J            269
748           BBB             2013B            240
2515          BBB             2013J            268
4752          BBB             2014B            234
6365          BBB             2014J            262
8657          CCC             2014B            241
10593         CCC             2014J            269


## 4. Registration and withdrawal

We attach registration dates from `studentRegistration.csv` and define `is_withdrawn` when `date_unregistration` is present. A high withdrawal rate in OULAD is expected; we keep this flag for interpretation in later notebooks, not as a clustering input at this stage.



In [32]:
reg_cols = STUDENT_KEYS + ["date_registration", "date_unregistration"]
master = master.merge(student_reg[reg_cols], on=STUDENT_KEYS, how="left")
master["is_withdrawn"] = master["date_unregistration"].notna().astype(int)

withdrawn_rate = master["is_withdrawn"].mean()
print(f"Withdrawal rate: {withdrawn_rate:.1%} ({master['is_withdrawn'].sum():,} students)")


Withdrawal rate: 30.9% (10,072 students)


**Interpretation**

The withdrawal rate we got above is typical for OULAD: many students unregister before the module ends. We retain `is_withdrawn` and registration dates for later validation against clusters, alongside `final_result`.


## 5. Virtual Learning Environment (VLE) behavior

We summarize how often each student interacted with the platform: click totals, number of active days, and first/last access. Module presentations run between about 234 and 269 days; after aggregation we scale counts by `module_length` so engagement rates are comparable across modules.



### 5.1 Linking click logs to material types

Before aggregating, we join each `studentVle` row to `vle.csv` on `id_site` so every click carries an `activity_type` (quiz, forum, resource, etc.). The preview below lets us confirm the join succeeded and see how click volume splits across activity types.


In [33]:
# Preview: studentVle + activity_type (full aggregation in next cells)
vle_preview = student_vle.merge(
    vle[["id_site", "code_module", "code_presentation", "activity_type"]],
    on=["id_site", "code_module", "code_presentation"],
    how="left",
)
activity_preview = (
    vle_preview.groupby(STUDENT_KEYS + ["activity_type"], observed=True)["sum_click"]
    .sum()
    .reset_index()
)
print(f"Activity-type rows (preview): {len(activity_preview):,}")
display(activity_preview.head(6))


Activity-type rows (preview): 240,357


,id_student,code_module,code_presentation,activity_type,sum_click
0,6516,AAA,2014J,dataplus,21
1,6516,AAA,2014J,forumng,451
2,6516,AAA,2014J,homepage,497
3,6516,AAA,2014J,oucontent,1505
4,6516,AAA,2014J,resource,31
5,6516,AAA,2014J,subpage,143


### 5.2 VLE aggregation (per student–module)

We aggregate `studentVle` at the level of each registration: total clicks, active days, first/last access, weekly click totals (weeks 0–33), and clicks split by activity type (quiz, forum, resource). Logic lives in `build_vle_aggregations()` so we can reuse it outside the notebook.

Weekly click columns go from `week_0` to `week_33` (capped at 34 to give a consistent shape across all module presentations — the longest modules run ≈38 weeks but most activity in later weeks is near zero).


In [34]:
vle_agg = build_vle_aggregations(student_vle, vle)
print(f"VLE aggregation shape: {vle_agg.shape}")
display(vle_agg.head(3))


VLE aggregation shape: (29228, 44)


,id_student,code_module,code_presentation,total_clicks,active_days,first_access_day,last_access_day,week_0_clicks,week_1_clicks,week_2_clicks,week_3_clicks,week_4_clicks,week_5_clicks,week_6_clicks,week_7_clicks,week_8_clicks,week_9_clicks,week_10_clicks,week_11_clicks,week_12_clicks,week_13_clicks,week_14_clicks,week_15_clicks,week_16_clicks,week_17_clicks,week_18_clicks,week_19_clicks,week_20_clicks,week_21_clicks,week_22_clicks,week_23_clicks,week_24_clicks,week_25_clicks,week_26_clicks,week_27_clicks,week_28_clicks,week_29_clicks,week_30_clicks,week_31_clicks,week_32_clicks,week_33_clicks,quiz_clicks,forum_clicks,resource_clicks
0,6516,AAA,2014J,2791,159,-23,269,229.00,42.00,79.00,193.00,69.00,34.00,10.00,93.00,57.00,61.00,125.00,46.00,2.00,3.00,18.00,33.00,76.00,17.00,151.00,111.00,38.00,0.00,44.00,103.00,173.00,87.00,36.00,90.00,49.00,90.00,55.00,54.00,119.00,79.00,0.00,451.00,31.00
1,8462,DDD,2013J,646,56,-6,118,81.00,146.00,9.00,23.00,79.00,18.00,63.00,17.00,10.00,0.00,23.00,5.00,52.00,17.00,1.00,9.00,12.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,36.00,70.00
2,8462,DDD,2014J,10,1,10,10,0.00,10.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2.00,0.00


### 5.3 Weekly click profile

Each column `week_N_clicks` stores how many clicks occurred in that week of the module (day index // 7). These series support temporal features in notebook 02 (e.g. late vs early engagement).



In [35]:
week_cols = [c for c in vle_agg.columns if c.startswith("week_") and c.endswith("_clicks")]
print(f"Weekly click columns: {len(week_cols)} (week_0 … week_{MAX_WEEK})")
print(vle_agg[week_cols[:5]].describe().loc[["mean", "max"]])


Weekly click columns: 34 (week_0 … week_33)
      week_0_clicks  week_1_clicks  week_2_clicks  week_3_clicks  \
mean          68.87          61.50          78.93          56.92   
max        4,385.00       1,566.00       1,709.00       2,334.00   

      week_4_clicks  
mean          51.50  
max        2,034.00  


## 6. Merge VLE features onto the master table

We left-join VLE aggregates so every registration keeps a row. Missing values mean the student never appeared in `studentVle`; we set those fields to **0**, not the column mean, because zero activity is informative for clustering.



In [36]:
master = master.merge(vle_agg, on=STUDENT_KEYS, how="left")

vle_fill = [
    "total_clicks", "active_days", "first_access_day", "last_access_day",
    "quiz_clicks", "forum_clicks", "resource_clicks",
] + week_cols

master[vle_fill] = master[vle_fill].fillna(0)
zero_vle = (master["total_clicks"] == 0).sum()
print(f"Students with zero VLE rows (filled with 0): {zero_vle:,} ({zero_vle / len(master):.1%})")


Students with zero VLE rows (filled with 0): 3,365 (10.3%)


**Interpretation**

Registrations with zero VLE rows are students who never logged in. Treating them as 0 (instead of the cohort average) keeps disengagement visible to later clustering models.



## 7. Assessment behavior and non-submission

OULAD does not create a row when a student skips an assessment. We therefore count how many TMA/CMA items each module expects (Exams excluded here to focus on ongoing coursework), compare that to actual submissions, and define **`total_missing`** as the gap. Banked results from previous attempts are removed before computing scores and delays.



In [37]:
# Non-Exam assessments only for due-count and delays (documented choice)
assess_tma_cma = assessments[assessments["assessment_type"] != "Exam"].copy()
print(f"Assessments catalog (non-Exam): {len(assess_tma_cma):,} rows")

sa = student_assess.merge(
    assess_tma_cma[["id_assessment", "code_module", "code_presentation", "date", "weight"]],
    on="id_assessment",
    how="inner",
)
sa = sa[sa["is_banked"] != 1].copy()
sa["submission_delay"] = sa["date_submitted"] - sa["date"]
print(f"Submissions after dropping banked: {len(sa):,}")
display(sa[["id_student", "id_assessment", "score", "submission_delay"]].head(3))


Assessments catalog (non-Exam): 182 rows
Submissions after dropping banked: 167,044


,id_student,id_assessment,score,submission_delay
0,11391,1752,78.00,-1.00
1,28400,1752,70.00,3.00
2,31604,1752,72.00,-2.00


## 8. Assessment aggregates

We compute per-student submission counts, weighted average score, delay statistics, and missing submissions. Rows with `is_banked = 1` are excluded from score and delay calculations so retake credits do not distort current-attempt behavior.



In [38]:
assess_agg = build_assessment_aggregations(student_assess, assessments, exclude_exam=True)
print(f"Assessment aggregation: {assess_agg.shape[0]:,} student-module rows")
display(assess_agg.head(3))


Assessment aggregation: 25,559 student-module rows


,id_student,code_module,code_presentation,total_submitted,score_std,avg_submission_delay,late_submission_count,early_submission_count,avg_score,total_assessments_due,total_missing
0,6516,AAA,2014J,5,10.33,-2.60,0,1,63.50,5,0
1,8462,DDD,2013J,3,5.03,-0.33,1,0,87.25,6,3
2,11391,AAA,2013J,5,3.08,-1.80,0,0,82.40,5,0


## 9. Final merges and rate normalization

Assessment features are merged on the same three keys. We then derive `clicks_per_day`, `active_day_rate`, and **`missing_rate`** (`total_missing / total_assessments_due`) by dividing where needed by `module_length`, so students in shorter and longer presentations are comparable.



In [39]:
master = master.merge(assess_agg, on=STUDENT_KEYS, how="left")

for col in [
    "total_assessments_due", "total_submitted", "total_missing",
    "avg_score", "score_std", "late_submission_count", "early_submission_count",
]:
    if col in master.columns:
        master[col] = master[col].fillna(0)

master["clicks_per_day"] = master["total_clicks"] / master["module_length"]
master["active_day_rate"] = master["active_days"] / master["module_length"]

due = master["total_assessments_due"].replace(0, np.nan)
master["missing_rate"] = (master["total_missing"] / due).fillna(0)

print("Added: clicks_per_day, active_day_rate, missing_rate")


Added: clicks_per_day, active_day_rate, missing_rate


> **Note on assessment fields for non-submitters:** `avg_score` and `score_std` are filled with **0** here when a student submitted no graded work — `avg_score = 0` means "no submissions recorded," not necessarily "scored zero."

> **`avg_submission_delay`:** Students with no submissions keep **NaN** in `master_raw.csv` (~7,000 rows). That does **not** mean they submitted on time. In notebook 02, `compute_group_d()` builds `submission_timing` by filling non-submitters with the global mean delay among actual submitters (≈ −10.9 days), then clipping — so clustering never treats them as deadline-on-time submitters.


## 10. Quality checks before export

We verify row count still matches `studentInfo`, inspect missing values, and review `final_result` and engagement extremes. These checks catch join mistakes before M2 builds features.



In [40]:
assert master.shape[0] == n_anchor, (
    f"Row count must match student_info: {n_anchor} vs {master.shape[0]}"
)

print(f"master_raw shape: {master.shape[0]:,} rows × {master.shape[1]} columns")
print(f"Columns ({len(master.columns)}): {list(master.columns)}")

missing = master.isna().sum()
print("\nColumns with any NaN:")
print(missing[missing > 0])

print("\nfinal_result distribution:")
print(master["final_result"].value_counts(normalize=True).mul(100).round(1).astype(str) + "%")

print(f"\nZero total_clicks: {(master['total_clicks'] == 0).sum():,}")
print(f"Any missing submission: {(master['total_missing'] > 0).sum():,}")


master_raw shape: 32,593 rows × 68 columns
Columns (68): ['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'module_length', 'date_registration', 'date_unregistration', 'is_withdrawn', 'total_clicks', 'active_days', 'first_access_day', 'last_access_day', 'week_0_clicks', 'week_1_clicks', 'week_2_clicks', 'week_3_clicks', 'week_4_clicks', 'week_5_clicks', 'week_6_clicks', 'week_7_clicks', 'week_8_clicks', 'week_9_clicks', 'week_10_clicks', 'week_11_clicks', 'week_12_clicks', 'week_13_clicks', 'week_14_clicks', 'week_15_clicks', 'week_16_clicks', 'week_17_clicks', 'week_18_clicks', 'week_19_clicks', 'week_20_clicks', 'week_21_clicks', 'week_22_clicks', 'week_23_clicks', 'week_24_clicks', 'week_25_clicks', 'week_26_clicks', 'week_27_clicks', 'week_28_clicks', 'week_29_clicks', 'week_30_clicks', 'week_31_clicks', 'week_32_clicks', 'week_33_clicks', 'quiz

## 11. Save deliverable

The table below is written to `data/processed/master_raw.csv` — the main handoff file for exploratory analysis (notebook 01) and feature engineering (notebook 02).



In [41]:
out_path = save_master_raw(master, PROCESSED / "master_raw.csv")
print(f"Saved: {out_path}")
print(f"{master.shape[0]:,} rows × {master.shape[1]} columns")


Saved: D:\Ensia\3rd year\Afaf\S2\Machine Learning\project\data\processed\master_raw.csv
32,593 rows × 68 columns


## 12. Summary and next steps

We produced **`master_raw.csv`**: one row per registration with demographics, module length, registration/withdrawal fields, VLE aggregates (including weekly and activity-type splits), and assessment metrics. Students with no VLE history remain at zero engagement — we did not mean-impute those values.




In [42]:
# ── Final column inventory ──────────────────────────────────────────────────
col_groups = {
    "Identifiers (drop before X)": [
        "id_student", "code_module", "code_presentation", "final_result"
    ],
    "Background (studentInfo)": [
        "gender", "region", "highest_education", "imd_band", "age_band",
        "num_of_prev_attempts", "studied_credits", "disability"
    ],
    "Module metadata": ["module_length"],
    "Registration (studentRegistration)": [
        "date_registration", "date_unregistration", "is_withdrawn"
    ],
    "VLE engagement (studentVle + vle)": [
        "total_clicks", "active_days", "first_access_day", "last_access_day",
        "quiz_clicks", "forum_clicks", "resource_clicks",
        "clicks_per_day", "active_day_rate",
        "week_N_clicks (week_0 to week_33)"
    ],
    "Assessment behavior (studentAssessment + assessments)": [
        "total_assessments_due", "total_submitted", "total_missing",
        "avg_score", "score_std", "late_submission_count",
        "early_submission_count", "avg_submission_delay", "missing_rate"
    ],
}

print(f"master_raw.csv — {master.shape[0]:,} rows × {master.shape[1]} columns\n")
for group, cols in col_groups.items():
    present = [c for c in cols if c in master.columns or "week_N" in c]
    print(f"  {group}")
    for c in present:
        if "week_N" in c:
            week_cols = [x for x in master.columns if x.startswith("week_") and x.endswith("_clicks")]
            print(f"    ↳ {len(week_cols)} weekly click columns: {week_cols[0]} … {week_cols[-1]}")
        else:
            print(f"    ↳ {c}")
    print()


master_raw.csv — 32,593 rows × 68 columns

  Identifiers (drop before X)
    ↳ id_student
    ↳ code_module
    ↳ code_presentation
    ↳ final_result

  Background (studentInfo)
    ↳ gender
    ↳ region
    ↳ highest_education
    ↳ imd_band
    ↳ age_band
    ↳ num_of_prev_attempts
    ↳ studied_credits
    ↳ disability

  Module metadata
    ↳ module_length

  Registration (studentRegistration)
    ↳ date_registration
    ↳ date_unregistration
    ↳ is_withdrawn

  VLE engagement (studentVle + vle)
    ↳ total_clicks
    ↳ active_days
    ↳ first_access_day
    ↳ last_access_day
    ↳ quiz_clicks
    ↳ forum_clicks
    ↳ resource_clicks
    ↳ clicks_per_day
    ↳ active_day_rate
    ↳ 34 weekly click columns: week_0_clicks … week_33_clicks

  Assessment behavior (studentAssessment + assessments)
    ↳ total_assessments_due
    ↳ total_submitted
    ↳ total_missing
    ↳ avg_score
    ↳ score_std
    ↳ late_submission_count
    ↳ early_submission_count
    ↳ avg_submission_delay
   